In [2]:
import pandas as pd
import json
import os
import glob
import re

# =============================================================================
# Параметры
# =============================================================================
INPUT_DIR = "../data/output"
DICT_PATH = "../data/dictionary/result/dictionary_en_ru.json"
OUTPUT_DIR = "../data/output"
FILE_PATTERN = "guns_result_*.csv"

# =============================================================================
# 1. Поиск самого свежего файла пушек
# =============================================================================
def get_latest_guns_file(directory, pattern):
    search_path = os.path.join(directory, pattern)
    files = glob.glob(search_path)
    if not files:
        raise FileNotFoundError(f"Не найдено файлов, соответствующих шаблону {search_path}")
    files.sort()
    latest = files[-1]
    print(f"Выбран файл: {os.path.basename(latest)}")
    return latest

# =============================================================================
# 2. Загрузка словаря и построение двух индексов
# =============================================================================
def clean_text(text):
    """Удаляет все теги вида [xxx]...[/xxx], оставляя только содержимое."""
    if not isinstance(text, str):
        return ""
    # Удаляем закрывающие теги [/...]
    t = re.sub(r'\[/[^\]]+\]', '', text)
    # Удаляем открывающие теги [...]
    t = re.sub(r'\[([^\]]+)\]', '', t)
    return t

def extract_prefix(text):
    """Возвращает часть строки до первого ' - ' (пробел-тире-пробел). Если ' - ' нет, всю строку."""
    parts = text.split(" - ", 1)
    return parts[0].strip()

def load_translation_dicts(json_path):
    """
    Загружает JSON-словарь и возвращает:
    - full_lookup: dict { lower(cleaned_en) : set(cleaned_ru) } для полного совпадения
    - prefix_lookup: dict { lower(en_prefix) : [ {cleaned_en, cleaned_ru, orig_en, orig_ru} ] }
    """
    with open(json_path, 'r', encoding='utf-8') as f:
        raw = json.load(f)
    
    full_lookup = {}
    prefix_lookup = {}
    
    for uid, entry in raw.items():
        en_orig = entry.get("en", "")
        ru_orig = entry.get("ru", "")
        if not en_orig:
            continue
        
        en_clean = clean_text(en_orig)
        ru_clean = clean_text(ru_orig) if ru_orig else ""
        
        # --- Индекс для полного совпадения (Name, Red Text, Drop Source) ---
        if en_clean.strip():
            key_full = en_clean.strip().lower()
            if ru_clean.strip():
                full_lookup.setdefault(key_full, set()).add(ru_clean.strip())
        
        # --- Индекс для поиска по префиксу (Legendary Effect) ---
        en_prefix = extract_prefix(en_clean)
        if en_prefix:
            key_prefix = en_prefix.lower()
            prefix_lookup.setdefault(key_prefix, []).append({
                "en_clean": en_clean,
                "ru_clean": ru_clean,
                "en_orig": en_orig,
                "ru_orig": ru_orig
            })
    
    print(f"Загружено уникальных полных ключей: {len(full_lookup)}")
    print(f"Загружено уникальных префиксов: {len(prefix_lookup)}")
    return full_lookup, prefix_lookup

# =============================================================================
# 3. Функции перевода
# =============================================================================
def translate_full(text, full_lookup):
    """Перевод по полному совпадению очищенного текста."""
    if not isinstance(text, str) or not text.strip():
        return ""
    query = text.strip().lower()
    candidates = full_lookup.get(query, set())
    if not candidates:
        return "(перевод не найден)"
    elif len(candidates) > 1:
        return "(требуется ручная проверка)"
    else:
        return next(iter(candidates))

def translate_legendary_clean(text, prefix_lookup):
    """Перевод Legendary Effect по префиксу – очищенный RU (без тегов)."""
    if not isinstance(text, str) or not text.strip():
        return ""
    csv_prefix = extract_prefix(text)
    if not csv_prefix:
        return "(перевод не найден – пустой префикс)"
    key = csv_prefix.lower()
    matches = prefix_lookup.get(key, [])
    
    if not matches:
        return "(перевод не найден)"
    elif len(matches) > 1:
        return "(требуется ручная проверка)"
    else:
        ru = matches[0]["ru_clean"]
        return ru if ru else "(перевод пустой)"

def translate_legendary_raw(text, prefix_lookup):
    """Перевод Legendary Effect по префиксу – оригинальный RU (с тегами)."""
    if not isinstance(text, str) or not text.strip():
        return ""
    csv_prefix = extract_prefix(text)
    if not csv_prefix:
        return "(перевод не найден – пустой префикс)"
    key = csv_prefix.lower()
    matches = prefix_lookup.get(key, [])
    
    if not matches:
        return "(перевод не найден)"
    elif len(matches) > 1:
        return "(требуется ручная проверка)"
    else:
        ru = matches[0]["ru_orig"]
        return ru if ru else "(перевод пустой)"

def translate_drop_source(text, full_lookup):
    """Для Drop Source: разделение по запятой и перевод каждого элемента."""
    if not isinstance(text, str) or not text.strip():
        return ""
    parts = [p.strip() for p in text.split(",") if p.strip()]
    translated = [translate_full(p, full_lookup) for p in parts]
    return ", ".join(translated)

# =============================================================================
# Основной блок выполнения
# =============================================================================
try:
    # Поиск файла
    latest_file = get_latest_guns_file(INPUT_DIR, FILE_PATTERN)
    
    # Загрузка словарей
    print("Загрузка и индексация словаря...")
    full_lookup, prefix_lookup = load_translation_dicts(DICT_PATH)
    
    # Чтение CSV
    df = pd.read_csv(latest_file)
    print(f"Прочитано строк: {len(df)}")
    
    # Проверяем наличие обязательных колонок
    required_cols = ["Name", "Red Text", "Drop Source", "Legendary Effect"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise KeyError(f"В файле отсутствуют необходимые столбцы: {missing}")
    
    # Перевод столбцов
    print("Выполняю перевод...")
    df["Name_ru"] = df["Name"].apply(lambda x: translate_full(x, full_lookup))
    df["Red Text_ru"] = df["Red Text"].apply(lambda x: translate_full(x, full_lookup))
    df["Drop Source_ru"] = df["Drop Source"].apply(lambda x: translate_drop_source(x, full_lookup))
    
    # Для Legendary Effect – два столбца: очищенный и «сырой»
    df["Legendary Effect_ru"] = df["Legendary Effect"].apply(
        lambda x: translate_legendary_clean(x, prefix_lookup)
    )
    df["Legendary Effect_ru_raw"] = df["Legendary Effect"].apply(
        lambda x: translate_legendary_raw(x, prefix_lookup)
    )
    
    # Сохранение результата
    base_name = os.path.basename(latest_file)
    name_without_ext, ext = os.path.splitext(base_name)
    output_name = f"{name_without_ext}_with_ru.csv"
    output_path = os.path.join(OUTPUT_DIR, output_name)
    
    df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"Результат сохранён: {output_path}")
    
except Exception as e:
    print(f"Ошибка: {e}")

Выбран файл: guns_result_20260806_232446.csv
Загрузка и индексация словаря...
Загружено уникальных полных ключей: 60959
Загружено уникальных префиксов: 94853
Прочитано строк: 133
Выполняю перевод...
Результат сохранён: ../data/output/guns_result_20260806_232446_with_ru.csv
